# Data Reduction

## Objetivo

Crear el dataset final del Sprint con las variables necesarias para responder a las preguntas de:

- Marketing y estrategia comercial.
- Operaciones y gestión de inventario.
- Experiencia del cliente.
- KPI 1–4 definidos por el equipo.

La reducción se realiza principalmente mediante **selección de variables**. No se eliminan filas por valores nulos en las reseñas, porque esos alojamientos siguen siendo válidos para los análisis de Marketing y Operaciones.


## Estrategia acordada

- Mantener un único registro por `apartment_id`; la selección del registro más reciente ya se realizó en Data Cleaning.
- Conservar únicamente las variables necesarias para los análisis departamentales y los KPI.
- Mantener los nulos de las puntuaciones y de `rating_above_80`.
- Conservar `availability_30/60/90/365` para la pregunta de Operaciones.
- Conservar únicamente `occupancy_30` y `occupancy_rate_30` para los KPI mensuales.
- Mantener `has_availability_numeric` y eliminar la columna original `has_availability`.
- Eliminar `is_instant_bookable` y `is_instant_bookable_numeric`, porque no aportan información a los análisis definidos.
- Para el cálculo operativo del KPI, los días no disponibles se consideran días ocupados, siguiendo el criterio acordado con el equipo.
- La ocupación debe interpretarse como una estimación, ya que el dataset no permite distinguir entre reservas reales y fechas bloqueadas por el anfitrión.
- No agregar los datos en esta fase: cada perfil realizará sus agrupaciones en su notebook de análisis.


## 1. Importación de librerías

In [10]:
from pathlib import Path

import pandas as pd

## 2. Localización automática de la raíz del proyecto

La ruta absoluta del ordenador no se guarda ni se muestra. La función busca la carpeta `Equip_34` desde la ubicación actual del notebook.


In [11]:
def encontrar_raiz_proyecto(nombre_carpeta="Equip_34"):
    actual = Path.cwd()

    for carpeta in [actual] + list(actual.parents):
        if carpeta.name == nombre_carpeta:
            return carpeta

    raise FileNotFoundError(
        f"No se encontró la carpeta '{nombre_carpeta}'."
    )


raiz_proyecto = encontrar_raiz_proyecto()

ruta_entrada = (
    raiz_proyecto
    / "Data"
    / "clean_dataset_29_06_2026.csv"
)

# El resultado final sobrescribe el mismo archivo Clean.
ruta_salida = ruta_entrada

print(
    "Archivo de entrada y salida:",
    f"Data/{ruta_entrada.name}",
)


Archivo de entrada y salida: Data/clean_dataset_29_06_2026.csv


## 3. Carga del dataset después de Data Cleaning y Data Transformation

El archivo se carga completamente en memoria antes de aplicar la reducción. Solo se sobrescribe al final, después de superar todas las validaciones.


In [12]:
df_original = pd.read_csv(ruta_entrada)

df = df_original.copy()

print("Dimensiones iniciales:", df.shape)
df.head()


Dimensiones iniciales: (6733, 46)


,apartment_id,name,description,host_id,neighbourhood_name,neighbourhood_district,room_type,accommodates,bathrooms,bedrooms,...,occupancy_30,occupancy_rate_30,occupancy_60,occupancy_rate_60,occupancy_90,occupancy_rate_90,occupancy_365,occupancy_rate_365,is_instant_bookable_numeric,has_availability_numeric
0,13707226,"Remarkable Value, Unbeatable Location",A spacious double bedroom with a balcony. It's...,80008404,el Barri G�tic,Ciutat Vella,Private room,2,4.0,1.0,...,0,0.00,0,0.0,0,0.00,185,50.68,1,NaN
1,13011987,"Lovely flat in Barcelona, 10' from the city ce...","Ideal for couples, located in a quiet and safe...",31321818,Sant Mart� de Proven�als,Sant Mart�,Entire home/apt,2,1.0,1.0,...,30,100.00,60,100.0,90,100.00,345,94.52,0,NaN
2,14999488,Habitaci�n doble en bonito apartamento,Piso grande y bonito de 90 m2. Tranquilo y por...,7093663,el Camp d'en Grassot i Gr�cia Nova,Gr�cia,Private room,2,1.0,1.0,...,26,86.67,42,70.0,42,46.67,261,71.51,1,NaN
3,6547870,Bella Vista. N� de registro: HUTB005799,Bella Vista es un apartamento �nico en Barcelo...,34249903,el Baix Guinard�,Horta-Guinard�,Entire home/apt,7,1.0,3.0,...,0,0.00,15,25.0,43,47.78,127,34.79,1,NaN
4,15253506,Casa de Andrea,"My house is very big, from the twenties. I onl...",17925327,Sants,Sants-Montju�c,Entire home/apt,14,2.0,5.0,...,19,63.33,24,40.0,25,27.78,27,7.40,0,NaN


## 4. Selección de variables necesarias


In [13]:
columnas_reducidas = [
    # Identificación y segmentación
    "apartment_id",
    "city",
    "room_type",

    # Marketing
    "price_€",

    # Operaciones
    "availability_30",
    "availability_60",
    "availability_90",
    "availability_365",
    "has_availability_numeric",

    # KPI mensual de ocupación estimada
    "occupancy_30",
    "occupancy_rate_30",

    # Experiencia del cliente
    "review_scores_rating",
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
    "rating_above_80",
]

columnas_faltantes = [
    columna
    for columna in columnas_reducidas
    if columna not in df.columns
]

if columnas_faltantes:
    raise ValueError(
        "Faltan columnas necesarias para Data Reduction: "
        f"{columnas_faltantes}"
    )

print("Número de columnas seleccionadas:", len(columnas_reducidas))


Número de columnas seleccionadas: 18


## 5. Validaciones antes de reducir

Estas comprobaciones no repiten Data Cleaning ni Data Transformation. Solo verifican que su resultado es apto para generar el dataset final.


In [14]:
duplicados_id = df["apartment_id"].duplicated().sum()

print("Apartment ID duplicados:", duplicados_id)

if duplicados_id > 0:
    raise ValueError(
        "El dataset contiene apartment_id duplicados. "
        "Debe revisarse Data Cleaning."
    )

limites_disponibilidad = {
    "availability_30": 30,
    "availability_60": 60,
    "availability_90": 90,
    "availability_365": 365,
}

for columna, limite in limites_disponibilidad.items():
    valores_validos = (
        df[columna].isna()
        | df[columna].between(0, limite)
    )

    if not valores_validos.all():
        raise ValueError(
            f"{columna} contiene valores fuera de 0–{limite}."
        )

rating_valido = (
    df["review_scores_rating"].isna()
    | df["review_scores_rating"].between(0, 100)
)

if not rating_valido.all():
    raise ValueError(
        "review_scores_rating contiene valores fuera de 0–100."
    )

subpuntuaciones = [
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
]

for columna in subpuntuaciones:
    valores_validos = (
        df[columna].isna()
        | df[columna].between(0, 10)
    )

    if not valores_validos.all():
        raise ValueError(
            f"{columna} contiene valores fuera de 0–10."
        )

ocupacion_valida = (
    df["occupancy_30"].isna()
    | df["occupancy_30"].between(0, 30)
)

if not ocupacion_valida.all():
    raise ValueError(
        "occupancy_30 contiene valores fuera de 0–30."
    )

tasa_ocupacion_valida = (
    df["occupancy_rate_30"].isna()
    | df["occupancy_rate_30"].between(0, 100)
)

if not tasa_ocupacion_valida.all():
    raise ValueError(
        "occupancy_rate_30 contiene valores fuera de 0–100."
    )

print("Validaciones superadas.")


Apartment ID duplicados: 0
Validaciones superadas.


## 6. Aplicación de Data Reduction


In [15]:
df_reduced = df[columnas_reducidas].copy()

# Booleano nullable: True, False o <NA>.
df_reduced["rating_above_80"] = (
    df_reduced["rating_above_80"]
    .astype("boolean")
)

# La versión numérica se mantiene como variable nullable.
df_reduced["has_availability_numeric"] = (
    pd.to_numeric(
        df_reduced["has_availability_numeric"],
        errors="coerce",
    )
    .astype("Int64")
)

print("Dimensiones originales:", df.shape)
print("Dimensiones finales:", df_reduced.shape)

reduccion_columnas_pct = round(
    (1 - df_reduced.shape[1] / df.shape[1]) * 100,
    2,
)

print(
    "Reducción del número de columnas:",
    f"{reduccion_columnas_pct}%",
)

df_reduced.head()


Dimensiones originales: (6733, 46)
Dimensiones finales: (6733, 18)
Reducción del número de columnas: 60.87%


,apartment_id,city,room_type,price_€,availability_30,availability_60,availability_90,availability_365,has_availability_numeric,occupancy_30,occupancy_rate_30,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,rating_above_80
0,13707226,barcelona,Private room,600.0,30,60,90,180,<NA>,0,0.00,94.0,10.0,9.0,9.0,9.0,9.0,True
1,13011987,barcelona,Entire home/apt,650.0,0,0,0,20,<NA>,30,100.00,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
2,14999488,barcelona,Private room,250.0,4,18,48,104,<NA>,26,86.67,90.0,10.0,10.0,10.0,9.0,10.0,True
3,6547870,barcelona,Entire home/apt,750.0,30,45,47,238,<NA>,0,0.00,86.0,9.0,8.0,9.0,9.0,9.0,True
4,15253506,barcelona,Entire home/apt,3150.0,11,36,65,338,<NA>,19,63.33,NaN,NaN,NaN,NaN,NaN,NaN,<NA>


## 7. Comprobación de cobertura por perfil

No se eliminan globalmente los alojamientos sin precio o sin puntuación. Cada análisis utilizará únicamente los registros válidos para su propia métrica.


In [16]:
cobertura = pd.DataFrame(
    {
        "perfil": [
            "Marketing",
            "Operaciones",
            "Experiencia del cliente",
        ],
        "registros_utilizables": [
            df_reduced[
                [
                    "city",
                    "room_type",
                    "price_€",
                ]
            ].dropna().shape[0],

            df_reduced[
                [
                    "city",
                    "availability_30",
                    "availability_60",
                    "availability_90",
                    "availability_365",
                ]
            ].dropna().shape[0],

            df_reduced[
                [
                    "city",
                    "review_scores_rating",
                ]
            ].dropna().shape[0],
        ],
    }
)

cobertura


,perfil,registros_utilizables
0,Marketing,6612
1,Operaciones,6733
2,Experiencia del cliente,5459


## 8. Validación final


In [17]:
if len(df_reduced) != len(df):
    raise ValueError(
        "Data Reduction ha modificado el número de filas."
    )

if not df_reduced["apartment_id"].is_unique:
    raise ValueError(
        "El dataset final contiene apartment_id duplicados."
    )

if list(df_reduced.columns) != columnas_reducidas:
    raise ValueError(
        "Las columnas finales no coinciden con la selección definida."
    )

print(
    "Filas conservadas:",
    len(df_reduced),
    "de",
    len(df),
)

print(
    "Apartment ID único:",
    df_reduced["apartment_id"].is_unique,
)

print(
    "Nulos en rating_above_80:",
    df_reduced["rating_above_80"].isna().sum(),
)

print("Distribución de rating_above_80:")

display(
    df_reduced["rating_above_80"]
    .value_counts(dropna=False)
)

df_reduced.info()


Filas conservadas: 6733 de 6733
Apartment ID único: True
Nulos en rating_above_80: 1274
Distribución de rating_above_80:


rating_above_80
True     4895
<NA>     1274
False     564
Name: count, dtype: Int64

<class 'pandas.DataFrame'>
RangeIndex: 6733 entries, 0 to 6732
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   apartment_id                 6733 non-null   int64  
 1   city                         6733 non-null   str    
 2   room_type                    6733 non-null   str    
 3   price_€                      6612 non-null   float64
 4   availability_30              6733 non-null   int64  
 5   availability_60              6733 non-null   int64  
 6   availability_90              6733 non-null   int64  
 7   availability_365             6733 non-null   int64  
 8   has_availability_numeric     6199 non-null   Int64  
 9   occupancy_30                 6733 non-null   int64  
 10  occupancy_rate_30            6733 non-null   float64
 11  review_scores_rating         5459 non-null   float64
 12  review_scores_accuracy       5450 non-null   float64
 13  review_scores_cleanliness    

## 9. Exportación del dataset final

El resultado de Data Reduction sobrescribe `clean_dataset_29_06_2026.csv`.

De este modo, la carpeta `Data` conserva el dataset raw original y el dataset clean final después de Data Cleaning, Data Transformation y Data Reduction. No se crea una copia adicional dentro de `Scripts`.


In [18]:
df_reduced.to_csv(
    ruta_salida,
    index=False,
    encoding="utf-8",
)

print(
    "Dataset final guardado correctamente en:",
    f"Data/{ruta_salida.name}",
)


Dataset final guardado correctamente en: Data/clean_dataset_29_06_2026.csv


## Resultado

`clean_dataset_29_06_2026.csv` queda como dataset final del Sprint después de aplicar Data Cleaning, Data Transformation y Data Reduction.

El dataset conserva un alojamiento por fila y únicamente las variables necesarias para los tres perfiles de análisis y los KPI definidos.

La ocupación se mantiene únicamente para el horizonte mensual de 30 días.

Siguiendo el criterio acordado con el equipo, los días no disponibles se consideran días ocupados para el cálculo de:

- `occupancy_30 = 30 - availability_30`
- `occupancy_rate_30 = occupancy_30 / 30 * 100`

No obstante, estas variables deben interpretarse como una **estimación de ocupación**, ya que el dataset no permite distinguir entre reservas reales y fechas bloqueadas por el anfitrión.
